# Lecture 19: Capstone & Future Trends — Answer Key
### NLP Course 2027 — Week 10

---
Complete answers for all four exercises in Lecture 19.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report
print('Ready')

---
## Exercise 19.1 — RAG Pipeline

**Task:** Build a full RAG pipeline using `sentence-transformers` for retrieval.

**Key concept:** RAG = Retriever + Generator. The retriever finds relevant passages from a knowledge base; the generator conditions its answer on those passages. This grounds the LLM in verifiable facts and reduces hallucination.

In [ ]:
knowledge_base = [
    'BERT is a bidirectional transformer model trained on masked language modeling and next sentence prediction.',
    'Fine-tuning means taking a pretrained model and training it on a task-specific labeled dataset for 2-5 epochs.',
    'Named Entity Recognition (NER) identifies named entities such as people, organizations, and places in text.',
    'TF-IDF weights terms by their frequency in the document divided by their frequency across all documents.',
    'Word2Vec learns word embeddings: king - man + woman = queen is the classic analogy example.',
    'The transformer attention mechanism allows each token to attend to all other tokens simultaneously.',
    'spaCy is a production-ready NLP library providing tokenization, POS tagging, NER, and dependency parsing.',
    'NLTK (Natural Language Toolkit) is an educational NLP library with access to corpora and text algorithms.',
    'Sentiment analysis classifies text as positive, negative, or neutral using Naive Bayes, LR, or BERT.',
    'RAG (Retrieval-Augmented Generation) combines a retriever with a generator LLM for grounded answers.',
    'LoRA trains low-rank update matrices instead of all model weights, reducing trainable params by 99%.',
    'WordPiece tokenization splits unknown words into subwords: unbelievable -> un ##believ ##able.',
    'Chunking groups POS-tagged words into noun phrases (NP) and verb phrases (VP).',
    'BM25 is a ranking function for information retrieval using term frequency and inverse document frequency.',
    'LDA (Latent Dirichlet Allocation) discovers latent topics as distributions over words in a corpus.',
]

try:
    from sentence_transformers import SentenceTransformer

    print('Loading sentence-transformers model...')
    embedder = SentenceTransformer('all-MiniLM-L6-v2')

    kb_embeddings = embedder.encode(knowledge_base, convert_to_numpy=True)

    def retrieve(query, k=3):
        q_emb  = embedder.encode([query], convert_to_numpy=True)
        scores = cosine_similarity(q_emb, kb_embeddings).flatten()
        top_k  = np.argsort(scores)[::-1][:k]
        return [(knowledge_base[i], scores[i]) for i in top_k]

    def build_rag_prompt(query, retrieved_docs):
        context = '\n'.join(f'  [{i+1}] {doc}' for i, (doc, _) in enumerate(retrieved_docs))
        return (
            f'Answer the question using ONLY the context below.\n\n'
            f'Context:\n{context}\n\n'
            f'Question: {query}\n'
            f'Answer:'
        )

    test_questions = [
        'What is the difference between BERT and Word2Vec?',
        'How does RAG work?',
        'What is LoRA used for?',
        'How does tokenization handle unknown words?',
    ]

    for q in test_questions:
        docs   = retrieve(q, k=3)
        prompt = build_rag_prompt(q, docs)
        print(f'Q: {q}')
        print('Retrieved passages:')
        for doc, score in docs:
            print(f'  (sim={score:.3f}) {doc[:80]}...')
        print()

except ImportError:
    print('sentence-transformers not installed. Using TF-IDF fallback retriever.')

    vec = TfidfVectorizer(stop_words='english')
    kb_vecs = vec.fit_transform(knowledge_base)

    def retrieve_tfidf(query, k=3):
        qvec   = vec.transform([query])
        scores = cosine_similarity(qvec, kb_vecs).flatten()
        top_k  = np.argsort(scores)[::-1][:k]
        return [(knowledge_base[i], scores[i]) for i in top_k]

    for q in ['What is BERT?', 'How does RAG work?', 'What is LoRA?']:
        print(f'Q: {q}')
        for doc, score in retrieve_tfidf(q):
            print(f'  (sim={score:.3f}) {doc[:80]}')
        print()

---
## Exercise 19.2 — Fairness Audit

**Task:** Test a sentiment classifier across 10 demographic groups using template-based probing. Flag groups with >5% deviation.

**Key concept:** Demographic parity requires that a classifier's positive prediction rate be equal across groups. Template-based testing (swapping demographic terms in identical sentences) isolates the model's bias from real content differences.

In [ ]:
try:
    from transformers import pipeline
    sentiment = pipeline('text-classification',
                         model='distilbert-base-uncased-finetuned-sst-2-english')
    model_available = True
except Exception:
    model_available = False

templates = [
    'The {group} employee did an excellent job on this project.',
    'The {group} student performed very well on the exam.',
    'The {group} manager made a great decision today.',
    'The {group} doctor treated the patient very effectively.',
    'The {group} engineer solved the problem brilliantly.',
]

demographic_groups = [
    'male', 'female', 'young', 'elderly',
    'White', 'Black', 'Asian', 'Latino', 'Muslim', 'Christian',
]

def audit_fairness(groups, templates, model_fn):
    results = {}
    for group in groups:
        texts = [t.format(group=group) for t in templates]
        preds = [model_fn(t)[0] for t in texts]
        pos_rate = sum(1 for p in preds if p['label'] == 'POSITIVE') / len(preds)
        results[group] = pos_rate
    return results

if model_available:
    group_rates = audit_fairness(demographic_groups, templates, sentiment)
else:
    # Simulated results to illustrate the analysis
    np.random.seed(42)
    group_rates = {g: min(1.0, max(0.0, np.random.normal(0.85, 0.08))) for g in demographic_groups}
    group_rates['Muslim'] = 0.60  # simulate a biased model
    print('(Using simulated results — model not available)')

mean_pos_rate = np.mean(list(group_rates.values()))
threshold = 0.05

print(f'Mean positive prediction rate: {mean_pos_rate:.3f}')
print(f'Bias threshold: ±{threshold:.0%}')
print()
print(f'{"Group":<12} {"Pos Rate":>10} {"Δ from mean":>12} {"Flagged?"}')
print('-' * 48)
flagged = []
for group, rate in sorted(group_rates.items(), key=lambda x: -x[1]):
    delta   = rate - mean_pos_rate
    flag    = '⚠ BIAS' if abs(delta) > threshold else ''
    if abs(delta) > threshold:
        flagged.append(group)
    print(f'{group:<12} {rate:>10.3f} {delta:>+12.3f} {flag}')

print()
if flagged:
    print(f'Groups with potential bias: {flagged}')
    print('Recommended action: collect more balanced training data or apply re-weighting.')
else:
    print('No significant bias detected at the 5% threshold.')

---
## Exercise 19.3 — Prompt Engineering Comparison

**Task:** Compare zero-shot, few-shot, and chain-of-thought prompting on customer support tickets.

**Key concept:** Few-shot prompting provides in-context examples that guide the model's output format and reasoning. Chain-of-thought (CoT) prompting asks the model to reason step-by-step before answering — this helps on ambiguous inputs but adds latency.

In [ ]:
zero_shot_template = """Classify the sentiment of the following customer support ticket as POSITIVE, NEGATIVE, or NEUTRAL.
Only reply with one word.

Ticket: {text}
Sentiment:"""

few_shot_template = """Classify the sentiment of each support ticket as POSITIVE, NEGATIVE, or NEUTRAL.

Ticket: "My issue was resolved within minutes, great support!"
Sentiment: POSITIVE

Ticket: "I waited 3 hours and no one helped me. Unacceptable."
Sentiment: NEGATIVE

Ticket: "Received confirmation of my request."
Sentiment: NEUTRAL

Ticket: {text}
Sentiment:"""

cot_template = """Analyze this support ticket step by step, then classify sentiment.

Ticket: {text}

Step 1 - Identify emotional words:
Step 2 - Identify the outcome (resolved/unresolved/neutral):
Step 3 - Overall sentiment:
Final answer (POSITIVE/NEGATIVE/NEUTRAL):"""

test_tickets = [
    ('Your team solved my billing issue quickly!', 'POSITIVE'),
    ('Three weeks and still no response to my complaint.', 'NEGATIVE'),
    ('I received the tracking number for my order.', 'NEUTRAL'),
    ('This is absolutely the worst service I have ever experienced.', 'NEGATIVE'),
    ('Thank you for promptly addressing my refund request.', 'POSITIVE'),
    ('My account has been updated as requested.', 'NEUTRAL'),
    ('The technician was incredibly helpful and professional!', 'POSITIVE'),
    ('Nobody has called me back despite three voicemails.', 'NEGATIVE'),
    ('Ticket #12345 has been assigned to an agent.', 'NEUTRAL'),
    ('I am thrilled with the fast resolution!', 'POSITIVE'),
]

# Without an LLM API, we use a local TF-IDF classifier as a proxy
# to demonstrate the evaluation framework

# Simulate responses (in practice, call an LLM API for each prompt)
# Here we use a simple keyword heuristic to simulate three strategies
def simulate_zero_shot(text):
    pos = sum(1 for w in ['quickly', 'great', 'helpful', 'thrilled', 'promptly'] if w in text.lower())
    neg = sum(1 for w in ['worst', 'nobody', 'complaint', 'weeks', 'voicemail'] if w in text.lower())
    if pos > neg: return 'POSITIVE'
    if neg > pos: return 'NEGATIVE'
    return 'NEUTRAL'

def simulate_few_shot(text):   return simulate_zero_shot(text)  # same heuristic
def simulate_cot(text):        return simulate_zero_shot(text)  # same heuristic

print(f'{"Strategy":<12} {"Accuracy":>10}')
print('-' * 24)
for name, fn in [('Zero-shot', simulate_zero_shot),
                 ('Few-shot',  simulate_few_shot),
                 ('CoT',       simulate_cot)]:
    preds = [fn(t) for t, _ in test_tickets]
    true  = [l for _, l in test_tickets]
    acc   = accuracy_score(true, preds)
    print(f'{name:<12} {acc:>10.2%}')

print()
print('With a real LLM (GPT-4, Claude), expected accuracy comparison:')
print('  Zero-shot: ~80–85%  (works for clear-cut cases)')
print('  Few-shot:  ~88–92%  (examples clarify edge cases)')
print('  CoT:       ~90–93%  (best on ambiguous cases; adds ~200ms latency)')
print()
print('Prompt templates printed for reference:')
for name, tmpl in [('Zero-shot', zero_shot_template),
                   ('Few-shot',  few_shot_template),
                   ('CoT',       cot_template)]:
    print(f'\n--- {name} ---')
    print(tmpl.format(text='[TICKET TEXT HERE]'))

---
## Exercise 19.4 — Capstone: News Sentiment Analysis System

**Task:** Worked example capstone — full pipeline from data to deployment.

**Domain:** Financial news sentiment (positive/negative/neutral) — practical for investment decisions, risk monitoring, and automated trading signals.

In [ ]:
# === SECTION 1: Data ===
# Simulated financial news with labels
news_data = [
    ('Apple reports record quarterly profit beating analyst expectations', 'positive'),
    ('Fed raises interest rates signaling continued tightening', 'negative'),
    ('Tesla stock surges after strong delivery numbers', 'positive'),
    ('Bank of America warns of rising loan defaults amid recession fears', 'negative'),
    ('Inflation cools for third consecutive month easing pressure on consumers', 'positive'),
    ('Tech layoffs continue as Microsoft cuts 10000 jobs', 'negative'),
    ('OPEC agrees to maintain current oil production levels', 'neutral'),
    ('Amazon expands into healthcare with new telehealth service', 'positive'),
    ('Crypto market sees 20 percent drop following regulatory concerns', 'negative'),
    ('GDP growth slightly above forecast at 2.1 percent', 'neutral'),
    ('S&P 500 closes at all-time high on strong earnings season', 'positive'),
    ('Housing market slows as mortgage rates hit 20-year high', 'negative'),
    ('Federal Reserve holds rates steady in surprise decision', 'neutral'),
    ('Nvidia smashes revenue expectations driven by AI chip demand', 'positive'),
    ('China economic slowdown raises global growth concerns', 'negative'),
    ('Gold prices stabilize after weeks of volatility', 'neutral'),
    ('Unemployment falls to 50-year low of 3.4 percent', 'positive'),
    ('Supply chain disruptions persist threatening holiday sales', 'negative'),
    ('Euro zone GDP growth matches expectations at 0.3 percent', 'neutral'),
    ('Berkshire Hathaway posts strong returns amid market turbulence', 'positive'),
] * 3  # 60 examples

texts  = [t for t, _ in news_data]
labels = [l for _, l in news_data]

from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels)

print(f'Dataset: {len(texts)} examples | Train: {len(X_train)} | Test: {len(X_test)}')
from collections import Counter
print(f'Label distribution: {Counter(labels)}')

In [ ]:
# === SECTION 2: Three Model Experiments ===

# Experiment 1: TF-IDF + Logistic Regression (baseline)
pipe1 = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 2), min_df=1)),
    ('clf',   LogisticRegression(max_iter=500, C=1.0)),
])
pipe1.fit(X_train, y_train)
preds1 = pipe1.predict(X_test)
acc1   = accuracy_score(y_test, preds1)
print(f'Experiment 1 — TF-IDF + LR:  accuracy={acc1:.4f}')

# Experiment 2: Better features (char n-grams + word n-grams)
from sklearn.pipeline import FeatureUnion
from sklearn.preprocessing import FunctionTransformer
pipe2 = Pipeline([
    ('tfidf', TfidfVectorizer(ngram_range=(1, 3), min_df=1,
                              analyzer='word', sublinear_tf=True)),
    ('clf',   LogisticRegression(max_iter=1000, C=5.0)),
])
pipe2.fit(X_train, y_train)
preds2 = pipe2.predict(X_test)
acc2   = accuracy_score(y_test, preds2)
print(f'Experiment 2 — TF-IDF(1-3)+LR: accuracy={acc2:.4f}')

# Experiment 3: Transformer (FinBERT-style zero-shot classification)
try:
    from transformers import pipeline as hf_pipeline
    zsc = hf_pipeline('zero-shot-classification',
                      model='facebook/bart-large-mnli')
    label_map = {'positive': 'positive', 'negative': 'negative', 'neutral': 'neutral'}
    preds3 = []
    for text in X_test:
        result = zsc(text, list(label_map.keys()))
        preds3.append(result['labels'][0])
    acc3 = accuracy_score(y_test, preds3)
    print(f'Experiment 3 — Zero-shot BART:  accuracy={acc3:.4f}')
except Exception:
    print('Experiment 3 — Transformer: not available (install transformers)')
    acc3 = None

# Summary table
print()
print('Experiments Summary:')
experiments = [
    ('TF-IDF + LR (baseline)',       acc1),
    ('TF-IDF(1-3) + LR (improved)',  acc2),
    ('Zero-shot Transformer',         acc3 if acc3 else '(N/A)'),
]
for name, acc in experiments:
    acc_str = f'{acc:.4f}' if isinstance(acc, float) else acc
    print(f'  {name:<35}: {acc_str}')

In [ ]:
# === SECTION 3: Error Analysis ===
print('Error Analysis — Best model (TF-IDF + LR improved):')
print(classification_report(y_test, preds2, zero_division=0))

errors = [(X_test[i], y_test[i], preds2[i])
          for i in range(len(y_test)) if y_test[i] != preds2[i]]
if errors:
    print(f'\nMisclassified examples ({len(errors)} total):')
    for text, true, pred in errors[:3]:
        print(f'  Text: {text}')
        print(f'  True={true}, Predicted={pred}')
        print()
else:
    print('No errors on this test split (small dataset with duplicates).')

print('Common error patterns in financial NLP:')
patterns = [
    'Neutral/negative confusion: "stabilize", "holds steady" are neutral, not negative',
    'Sarcasm: "great news" can be sarcastic in financial journalism',
    'Domain knowledge: Fed rate hike is negative for bonds but possibly positive for banks',
    'Comparative: "beats expectations" is positive even if absolute numbers look bad',
]
for p in patterns:
    print(f'  • {p}')

In [ ]:
# === SECTION 4: Deployment Sketch ===
import pickle

# Save the best model
with open('/tmp/news_sentiment_model.pkl', 'wb') as f:
    pickle.dump(pipe2, f)
print('Model saved to /tmp/news_sentiment_model.pkl')

deployment_api = '''
from fastapi import FastAPI
from pydantic import BaseModel
import pickle

app = FastAPI(title="Financial News Sentiment API")

with open("news_sentiment_model.pkl", "rb") as f:
    model = pickle.load(f)

class NewsRequest(BaseModel):
    headline: str

@app.post("/predict")
def predict(req: NewsRequest):
    label = model.predict([req.headline])[0]
    proba = model.predict_proba([req.headline])[0]
    classes = model.classes_
    return {
        "headline": req.headline,
        "sentiment": label,
        "confidence": float(max(proba)),
        "scores": dict(zip(classes, proba.tolist()))
    }
'''

print('\nDeployment API sketch:')
print(deployment_api)

# Quick demo
demo_headlines = [
    'Apple reports record quarterly profit beating analyst expectations',
    'Bank warns of rising defaults as economy slows',
    'Federal Reserve holds rates steady',
]
print('Live predictions from best model:')
for h in demo_headlines:
    pred = pipe2.predict([h])[0]
    prob = pipe2.predict_proba([h]).max()
    print(f'  [{pred.upper():<8} {prob:.2f}] {h}')

---
## Summary

| Exercise | Key Concept |
|----------|-------------|
| 19.1 | RAG = retriever + generator; grounds LLM in verifiable facts, reduces hallucination |
| 19.2 | Demographic parity: equal positive rates across groups; template probing isolates bias |
| 19.3 | Few-shot > zero-shot > CoT by latency; CoT wins on ambiguous inputs |
| 19.4 | Full capstone: data → 3 experiments → error analysis → serialization → API |

---
*NLP Course 2027 — Week 10*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**